## Adds sentence_id to transaction table

Useful later on when ner and timex tags need to be added.

In [1]:
import sqlite3
import pandas as pd
import time
from tqdm import tqdm

### Variables from conf file

In [2]:
# database file path
DB_FILE = "../../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db"

# transaction table to update with status
TRANSACTION_TABLE = "transaction_row"
HEAD_TABLE = "transaction_head"


### Connect to database

In [3]:
# connecting with database
conn = sqlite3.connect(DB_FILE)
cur = conn.cursor()

### Add sentence_id to transaction table

In [4]:
def column_exists(cursor, table, column):
    cursor.execute(f"PRAGMA table_info({table})")
    return any(row[1] == column for row in cursor.fetchall())

In [5]:
start = time.time()

# add sentence_id column if it doesn't exist
if not column_exists(cur, TRANSACTION_TABLE, "sentence_id"):
    cur.execute("ALTER TABLE " + TRANSACTION_TABLE +  " ADD COLUMN sentence_id INTEGER")

# add data
cur.execute(f"""
UPDATE {TRANSACTION_TABLE}
SET sentence_id = (
    SELECT th.sentence_id
    FROM {HEAD_TABLE} th
    WHERE th.id = {TRANSACTION_TABLE}.head_id
)
WHERE EXISTS (
    SELECT 1
    FROM {HEAD_TABLE} th
    WHERE th.id = {TRANSACTION_TABLE}.head_id
);
"""
)

conn.commit()

end = time.time()

elapsed_time = end - start

minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)

print(f"time: {minutes} minute(s) and {seconds} second(s)")

time: 1 minute(s) and 50 second(s)


### Check the results (not part of final workflow)

In [6]:
query = f"SELECT * FROM transaction_row limit 10"

res = pd.read_sql(query, conn)
res

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos,sentence_id
0,1,2,3,-1,obl,lõpus,lõpp,"com,in,sg",None,S,3
1,2,2,5,1,nsubj,Türi,Türi,"gen,prop,sg",None,S,3
2,3,2,6,2,obl,1.,1.,"<?>,ord,roman",None,N,3
3,4,3,1,-3,obj,Bändi,bänd,"adit,com,sg",None,S,5
4,5,3,9,-2,nsubj,kidramees,kidramees,"com,nom,sg",None,S,5
5,6,3,10,-1,aux,ei,ei,"aux,neg",None,V,5
6,7,3,12,1,obl,keeltele,keel,"all,com,pl",None,S,5
7,8,3,13,2,compound:prt,pihta,pihta,,None,D,5
8,9,4,4,-2,nsubj,solist,solist,"com,nom,sg",None,S,5
9,10,4,5,-1,aux,ei,ei,"aux,neg",None,V,5


In [7]:
conn.close()